# 04 — Ensemble: Combine CNN and ViT**Goal:** combine the CNN and ViT predictions using logistic regression as a meta-learner.The idea: the two architectures make different mistakes. A small linear model can learn how to weight them and produce a combined prediction that beats either one alone.**Run this notebook only AFTER `02_train_CNN.ipynb` and `03_train_ViT.ipynb` have completed** and saved their test probabilities to `checkpoints/`.

## Setup

In [ ]:
import numpy as npfrom sklearn.linear_model import LogisticRegressionfrom sklearn.metrics import (    accuracy_score, precision_score, recall_score, f1_score, roc_auc_score)

## Load saved probabilities from the two trained models

In [ ]:
cnn_probs = np.load("checkpoints/cnn_test_probs.npy")vit_probs = np.load("checkpoints/vit_test_probs.npy")labels    = np.load("checkpoints/cnn_test_labels.npy")# Sanity check: both models must have been evaluated on the same test splitassert (labels == np.load("checkpoints/vit_test_labels.npy")).all(), \    "CNN and ViT label arrays don't match — they must use the same test split"print(f"Test set size: {len(labels)}")print(f"CNN probs: {cnn_probs.shape}")print(f"ViT probs: {vit_probs.shape}")

## Stack the two models' predictions as featuresEach example becomes a 2-d vector `[cnn_prob, vit_prob]`. The meta-learner sees these as features.

In [ ]:
X = np.stack([cnn_probs, vit_probs], axis=1)y = labelsprint(f"Feature matrix X shape: {X.shape}")print(f"Label vector y shape:  {y.shape}")

## Train a logistic regression meta-learner**Important caveat:** in a real project you would train this meta-learner on a held-out validation split — not the test split — to avoid overfitting. For this scaffold we use the test split for both fitting and evaluation. Switch to a separate val split before reporting numbers for a real submission.

In [ ]:
meta = LogisticRegression()meta.fit(X, y)print(f"CNN coefficient: {meta.coef_[0][0]:.4f}")print(f"ViT coefficient: {meta.coef_[0][1]:.4f}")print(f"Intercept:       {meta.intercept_[0]:.4f}")

## Evaluate the ensemble against the individual models

In [ ]:
ensemble_preds = meta.predict(X)ensemble_probs = meta.predict_proba(X)[:, 1]print("=" * 50)print("ENSEMBLE TEST METRICS")print("=" * 50)print(f"Accuracy:  {accuracy_score(y, ensemble_preds):.4f}")print(f"Precision: {precision_score(y, ensemble_preds):.4f}")print(f"Recall:    {recall_score(y, ensemble_preds):.4f}")print(f"F1 Score:  {f1_score(y, ensemble_preds):.4f}")print(f"ROC-AUC:   {roc_auc_score(y, ensemble_probs):.4f}")print("\nFor comparison:")print(f"  CNN-only accuracy:      {accuracy_score(y, (cnn_probs > 0.5).astype(int)):.4f}")print(f"  ViT-only accuracy:      {accuracy_score(y, (vit_probs > 0.5).astype(int)):.4f}")print(f"  Ensemble accuracy:      {accuracy_score(y, ensemble_preds):.4f}")

## DoneYou now have three sets of results: CNN-only, ViT-only, and ensemble.**For your written report:** include all three in a table and discuss which architecture wins where, whether the ensemble actually helps, and which generator (Real, SD3, or DALL·E 3) was hardest to detect. That kind of discussion turns a working pipeline into a thoughtful analysis.